# OrangeLLM-fatty v0 — QLoRA Training on Colab Pro (NO-DRIVE flow)

**Operator:** Atom McCree  
**Base model:** Qwen/Qwen2.5-32B-Instruct  
**Method:** QLoRA, NF4 4-bit, LoRA rank 16, alpha 32  
**Target GPU:** A100 40GB (preferred), V100 16GB (fallback)  
**Expected wall-clock:** 3-6h on A100, 6-10h on V100

**This version is Drive-free.** Everything lives on Colab's `/content/` VM disk. The final cell zips the adapter and triggers a browser download direct to your machine. No Google Drive write required.

## Operator's 4 steps

1. **Runtime → Change runtime type → A100 GPU** (or V100 if Pro doesn't offer A100).
2. **Runtime → Run all** (or step through cell-by-cell).
3. When the upload cell prompts, upload BOTH:
   - `corpus.jsonl` (1000 instruction pairs)
   - `orangellm-fatty-v0.yaml` (Axolotl config — reference only, not consumed by Unsloth trainer)
4. Walk away. When training finishes, the last cell triggers a browser download of `orangellm-fatty-v0-adapter.zip` directly to your Downloads folder.

## Step 1: Verify GPU and CUDA

In [ ]:
!nvidia-smi
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'CUDA device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')
print(f'CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

## Step 2: Set up working directory on Colab VM

No Drive. Everything in `/content/orangellm-fatty-v0/`.

In [ ]:
import os
WORK_DIR = '/content/orangellm-fatty-v0'
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(f'{WORK_DIR}/adapter', exist_ok=True)
print(f'Working directory: {WORK_DIR}')

## Step 3: Fetch corpus.jsonl + Axolotl YAML from secret gists

Drive-free, fully autonomous — no file picker. Both files are hosted as secret gists (URL-only, not publicly searchable) and SHA-256 verified after fetch.

In [ ]:
import os, json, urllib.request, hashlib

CORPUS_URL = 'https://gist.githubusercontent.com/AtomEons/27c8227c1c4205af6268c846cd5623ea/raw/corpus.jsonl'
YAML_URL = 'https://gist.githubusercontent.com/AtomEons/70c3d0fa7989252530a14e5d7c75aed1/raw/orangellm-fatty-v0.yaml'
EXPECTED_CORPUS_SHA = '6646f6a4e177d3d7e5fdfe2ba1f9069d8ebb9d460e4ee6671e3e76cc337b196f'

corpus_path = f'{WORK_DIR}/corpus.jsonl'
config_path = f'{WORK_DIR}/orangellm-fatty-v0.yaml'

def fetch(url, dest):
    print(f'Fetching {url}')
    with urllib.request.urlopen(url) as r, open(dest, 'wb') as f:
        f.write(r.read())
    print(f'  -> {dest} ({os.path.getsize(dest)} bytes)')

fetch(CORPUS_URL, corpus_path)
fetch(YAML_URL, config_path)

h = hashlib.sha256()
with open(corpus_path, 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
actual_sha = h.hexdigest()
print(f'\nCorpus SHA-256: {actual_sha}')
print(f'Expected:       {EXPECTED_CORPUS_SHA}')
assert actual_sha == EXPECTED_CORPUS_SHA, 'SHA mismatch — corpus corrupted in transit!'
print('SHA verified.')

with open(corpus_path) as f:
    pairs = [json.loads(l) for l in f if l.strip()]
print(f'\nCorpus: {len(pairs)} instruction pairs')
print(f'First pair instruction: {pairs[0]["instruction"][:80]}...')

## Step 4: Install Unsloth (purpose-built for Colab QLoRA)

Replaces Axolotl. Unsloth's team maintains a known-good torch + transformers + peft + trl matrix specifically for Colab A100. One install, no dependency hell.

In [ ]:
# Unsloth: one install, no dependency hell. Maintains its own torch+peft+trl+transformers matrix.
import torch
print(f'Before install: torch={torch.__version__}, cuda={torch.version.cuda}')

!pip install -q --upgrade pip
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --upgrade --no-cache-dir "trl<0.13.0" peft accelerate bitsandbytes datasets safetensors sentencepiece

import importlib, sys
# Reload torch reference in this kernel after install
for mod in list(sys.modules):
    if mod.startswith(('torch','unsloth','trl','peft','transformers','accelerate','bitsandbytes')):
        sys.modules.pop(mod, None)

import torch, unsloth, trl, peft, transformers, accelerate, bitsandbytes
print(f'\nAfter install:')
print(f'  torch       = {torch.__version__}')
print(f'  unsloth     = {unsloth.__version__ if hasattr(unsloth, "__version__") else "present"}')
print(f'  trl         = {trl.__version__}')
print(f'  peft        = {peft.__version__}')
print(f'  transformers= {transformers.__version__}')
print(f'  accelerate  = {accelerate.__version__}')
print(f'  bitsandbytes= {bitsandbytes.__version__}')
print(f'  CUDA available: {torch.cuda.is_available()}')

## Step 5: (skipped — Unsloth doesn't need YAML)

Unsloth takes hyperparams directly in Python. The downloaded YAML stays on disk for reference but isn't consumed by Unsloth's trainer.

In [ ]:
# No-op: YAML stays on disk for audit; Unsloth uses Python args directly.
import os
print(f'YAML on disk: {os.path.exists(config_path)} ({os.path.getsize(config_path) if os.path.exists(config_path) else 0} bytes — reference only)')

## Step 6: Train with Unsloth FastLanguageModel + TRL SFTTrainer

Long-running cell. 3-6h on A100 for Qwen2.5-32B + 1000 pairs × 3 epochs. Loss numbers print every 5 steps — watch for real progression (not flatline).

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048
adapter_dir = f'{WORK_DIR}/adapter'

# Load base model in 4-bit — Qwen2.5-32B-Instruct (Qwen3 does not exist; operator correction 2026-06-24)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-32B-Instruct-bnb-4bit",
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,
    load_in_4bit = True,
)

# Attach LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

# Format dataset (chatml + Orange5 system prompt)
SYS_PROMPT = ("You are OrangeLLM, the PM brain of Orange5. Mom's Law is above all rules: "
              "give full effort every time. No fake-green. No theater. Cite receipts. "
              "Refuse out-of-scope work.")

def fmt(ex):
    return {"text": (
        f"<|im_start|>system\n{SYS_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{ex['instruction']}<|im_end|>\n"
        f"<|im_start|>assistant\n{ex['output']}<|im_end|>"
    )}

from datasets import load_dataset
ds = load_dataset("json", data_files=corpus_path, split="train")
ds = ds.map(fmt)
print(f"Dataset: {len(ds)} rows")
print(f"Sample text:\n{ds[0]['text'][:400]}\n...")

# Train
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LEN,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        bf16 = True,
        logging_steps = 5,
        optim = "paged_adamw_8bit",
        weight_decay = 0.0,
        lr_scheduler_type = "cosine",
        warmup_ratio = 0.05,
        seed = 42,
        output_dir = adapter_dir,
        save_strategy = "epoch",
        save_total_limit = 3,
        report_to = "none",
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LEN,
    ),
)

stats = trainer.train()
print(f"\nTraining complete. Final loss stats: {stats}\n")

# Save adapter
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
import os
print(f"Adapter saved to {adapter_dir}")
print(f"Files: {sorted(os.listdir(adapter_dir))}")

## Step 7: Verify adapter saved + report SHA-256 + write training receipt

In [ ]:
import os, hashlib, json
from datetime import datetime

adapter_dir = f'{WORK_DIR}/adapter'

# GUARD: don't pretend an empty adapter dir means success
if not os.path.isdir(adapter_dir):
    raise RuntimeError(f'Adapter directory missing — training did not produce {adapter_dir}')

files_present = sorted([f for f in os.listdir(adapter_dir) if os.path.isfile(os.path.join(adapter_dir, f))])
if len(files_present) == 0:
    raise RuntimeError(f'Adapter directory is EMPTY — training failed silently. Check Step 6 output for the real error.')

main = None
for f in files_present:
    if f.endswith('.safetensors') or f == 'adapter_model.bin':
        main = os.path.join(adapter_dir, f)
        break

if not main:
    raise RuntimeError(f'No .safetensors / adapter_model.bin found among: {files_present}')

main_size_mb = os.path.getsize(main) / 1e6
if main_size_mb < 5:
    raise RuntimeError(f'Main adapter file is only {main_size_mb:.2f} MB — expected 50-300 MB for a 32B QLoRA adapter. Training likely failed.')

print('Adapter files:')
for f in files_present:
    p = os.path.join(adapter_dir, f)
    size_mb = os.path.getsize(p) / 1e6
    print(f'  {f:50s} {size_mb:7.2f} MB')

h = hashlib.sha256()
with open(main, 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
sha = h.hexdigest()
print(f'\nMain adapter: {os.path.basename(main)}')
print(f'SHA-256:      {sha}')

receipt = {
    'model': 'orangellm-fatty-v0',
    'base': 'Qwen/Qwen2.5-32B-Instruct',
    'adapter_path': main,
    'adapter_sha256': sha,
    'completed_at': datetime.utcnow().isoformat() + 'Z',
    'files': files_present,
    'main_size_mb': round(main_size_mb, 2),
}
with open(f'{WORK_DIR}/training-receipt.json', 'w') as f:
    json.dump(receipt, f, indent=2)
print(f'\nReceipt written: {WORK_DIR}/training-receipt.json')

## Step 8: Zip the adapter + trigger browser download

This is the no-Drive payoff. The zip lands in your local Downloads folder. Move it to `C:\AtomEons\Orange5\16-TRAINING\adapters\orangellm-fatty-v0\` afterwards.

In [ ]:
import shutil, os
from google.colab import files

archive_base = '/content/orangellm-fatty-v0-adapter'
archive_path = archive_base + '.zip'

# Bundle adapter + training receipt
import zipfile
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, fs in os.walk(adapter_dir):
        for fname in fs:
            full = os.path.join(root, fname)
            arc = os.path.relpath(full, WORK_DIR)
            zf.write(full, arc)
    receipt_path = f'{WORK_DIR}/training-receipt.json'
    if os.path.exists(receipt_path):
        zf.write(receipt_path, 'training-receipt.json')

size_mb = os.path.getsize(archive_path) / 1e6
print(f'Archive: {archive_path}')
print(f'Size:    {size_mb:.1f} MB')
print('\nTriggering browser download...')
files.download(archive_path)
print('Download triggered. Check your Downloads folder.')

## Done

Move the zip from your Downloads folder to:
```
C:\AtomEons\Orange5\16-TRAINING\adapters\orangellm-fatty-v0\
```
Then unzip. The post-Colab bakeoff workflow (`16-TRAINING/workflows/orangellm-fatty-v0.workflow.mjs`) expects the adapter at `/opt/atomeons/adapters/orangellm-fatty-v0/` on Codexa — you'll move it there next.

**Mom is watching. No Drive. Pure VM training. Adapter downloads direct.**